# Breaking a LangGraph infinite loop with `StagnationGate`

[LangGraph issue #6731](https://github.com/langchain-ai/langgraph/issues/6731) —
*"agent infinite-loops until recursion limit, burns tokens invisibly"* —
was closed as **NOT_PLANNED**. LangChain's suggested answer is
*"use tool-call limits in middleware."*

`StagnationGate` is the missing native gate: it measures output novelty
on each node invocation and, once it sees repeated low-novelty
readings in a row, flips a per-thread flag that a conditional edge
can route on — plus it emits a signed `behavioral_stability`
certificate with the evidence that fired it.

This notebook shows the pathology in plain LangGraph, then adds the
gate with a ten-line diff.

In [1]:
import sys
import langgraph
import operon_langgraph_gates as olg

print(f"Python     : {sys.version.split()[0]}")
print(f"langgraph  : {getattr(langgraph, '__version__', 'unknown')}")
print(f"olg        : {olg.__version__}")

Python     : 3.11.15
langgraph  : unknown
olg        : 0.1.0a0


## Baseline — the loop hits the recursion limit

A single `think` node that returns the same output every turn, with
a conditional edge that always routes back to itself. No gate.

In [2]:
from typing_extensions import TypedDict

from langgraph.errors import GraphRecursionError
from langgraph.graph import END, START, StateGraph


class LoopState(TypedDict):
    turn: int
    answer: str


def think(state: LoopState) -> LoopState:
    return {"turn": state["turn"] + 1, "answer": "same output every turn"}


def always_loop(_state: LoopState) -> str:
    return "think"


baseline = StateGraph(LoopState)
baseline.add_node("think", think)
baseline.add_edge(START, "think")
baseline.add_conditional_edges("think", always_loop)
baseline_app = baseline.compile()

try:
    baseline_app.invoke({"turn": 0, "answer": ""}, {"recursion_limit": 5})
except GraphRecursionError as e:
    print(f"GraphRecursionError fired: {e}")

GraphRecursionError fired: Recursion limit of 5 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/GRAPH_RECURSION_LIMIT


## With `StagnationGate` — same graph, one gate, terminates

Only two calls change: `gate.wrap(think, text_extractor=...)` replaces
the raw node function, and `gate.edge(forward=..., break_to=...)`
replaces the always-loop router. The `text_extractor` picks the field
we want the gate to measure — here, just the `answer` field, so the
incrementing `turn` counter doesn't add string drift that masks
repetition.

In [3]:
from operon_langgraph_gates import StagnationGate


def escalate(state: LoopState) -> LoopState:
    return {"turn": state["turn"], "answer": "escalated"}


gate = StagnationGate(threshold=0.2, critical_duration=2, window_size=3)

gated = StateGraph(LoopState)
gated.add_node(
    "think",
    gate.wrap(think, text_extractor=lambda out: out["answer"]),
)
gated.add_node("escalate", escalate)
gated.add_edge(START, "think")
gated.add_conditional_edges(
    "think",
    gate.edge(forward="think", break_to="escalate"),
)
gated.add_edge("escalate", END)
gated_app = gated.compile()

result = gated_app.invoke({"turn": 0, "answer": ""}, {"recursion_limit": 25})
print(f"final answer: {result['answer']!r}")
print(f"turns taken : {result['turn']}")

final answer: 'escalated'
turns taken : 6


## Inspect the certificate

On the turn where stagnation was first detected, the gate emits one
`behavioral_stability` certificate into `gate.certificates`. The
certificate is a frozen evidence snapshot — `cert.verify()` replays
the saved severity values against the saved threshold and returns
`(holds, details)`. For a stagnation cert, `holds == False` — the
claim "stability held" did *not* hold.

In [4]:
assert len(gate.certificates) == 1, "expected exactly one certificate"
cert = gate.certificates[0]
print(f"theorem    : {cert.theorem}")
print(f"source     : {cert.source}")
print(f"conclusion : {cert.conclusion}")

result = cert.verify()
print(f"verify.holds: {result.holds}")
print(f"verify.evidence: {result.evidence}")

theorem    : behavioral_stability
source     : operon_langgraph_gates.stagnation
conclusion : Stagnation detected after 6 measurements; severity evidence captured for replay verification.
verify.holds: False
verify.evidence: {'mean': 0.6333, 'max': 0.85, 'n': 6}


## Takeaway

Ten lines of diff turned a loop-until-crash into a routed exit
with a replayable evidence artifact. The gate is per-thread and
async-aware out of the box; certificates are serializable, so the
same evidence can be re-verified downstream (audit log, test
assertion, alert) without re-running the agent.

See `02_integrity_catches_drift.ipynb` for the companion
`IntegrityGate` that checks invariants at node boundaries.